# ConvexPi — R starter

Write a quant strategy in **R** and submit it. The grader runs your R `on_day` over a hidden
out-of-sample period and scores it with the *same* engine as Python/Julia — so the same idea earns
the same OOS Sharpe in any language. The `convexpi` package gives you the market and one-call submit.

In [ ]:
# 1. Install the convexpi R package (market data + submit in one import).
install.packages("convexpi", repos = c("https://convexpi.r-universe.dev", "https://cloud.r-project.org"))
# latest dev instead: remotes::install_github("convexpi/convexpi-r")
library(convexpi)

In [ ]:
# 2. Load the exact market the grader uses (deterministic from the seed).
m <- synthetic_market("train")     # fit on "train"; you're scored on the hidden "test"
prices <- m$prices; features <- m$features
cat("prices:", dim(prices), "| features:", paste(names(features), collapse = ", "), "\n")

## 3. Write your strategy

Define `on_day(day, features, prices, portfolio)` returning target weights (one per stock). This is
exactly what the grader runs. Keep it as a string so you can both test it and submit it.

In [ ]:
strategy_code <- '
on_day <- function(day, features, prices, portfolio) {
  sig <- features[["mom_1m"]]          # 1-month momentum, z-scored across stocks
  sig[!is.finite(sig)] <- 0
  s <- sum(abs(sig))
  if (s > 0) sig / s else sig          # long winners / short losers, gross leverage 1
}'
eval(parse(text = strategy_code))      # define on_day locally so you can poke at it

In [ ]:
# 4. (optional) sanity-check on one in-sample day
d <- 300
w <- on_day(d, lapply(features, function(mat) mat[d, ]), prices[d, ], rep(0, ncol(prices)))
cat("weights sum to", sum(w), "over", length(w), "stocks\n")

## 5. Submit

Create an API key at **convexpi.ai/settings/api-keys** and set it below. Your OOS Sharpe appears on
the leaderboard within a few minutes.

In [ ]:
Sys.setenv(CONVEXPI_API_KEY = "cpk_...")   # <- your key
submit("my-r-momentum", strategy_code)      # slug defaults to "demo-fall-2026"